# Atom Training on Google Colab

Bootstrap, preflight, quick infra check, full run, and resume in one notebook.



## 1) Mount Drive


In [ ]:
from google.colab import drive
#drive.mount('/content/drive')
drive.mount("/content/drive", force_remount=True)


## 2) Configure variables

Defaults are set for your current public repo/branch, but values are only set if missing.



In [ ]:
import os

# Repo defaults (only applied when missing)
os.environ.setdefault('ATOM_REPO_URL', 'https://github.com/MBifolco/atom.git')
os.environ.setdefault('ATOM_BRANCH', 'main')

# Runtime/cache defaults
os.environ.setdefault('ATOM_DRIVE_REPO', '/content/drive/MyDrive/dev/atom')
os.environ.setdefault('ATOM_WORK_REPO', '/content/atom')
os.environ.setdefault('ATOM_INSTALL_JAX_CUDA', '1')
os.environ.setdefault('ATOM_JAX_VERSION', '0.7.2')
os.environ.setdefault('ATOM_DRIVE_REPO_SYNC_MODE', 'stash')  # stash|reset|skip_pull

# Training defaults
os.environ.setdefault('ATOM_USE_VMAP', '1')
os.environ.setdefault('ATOM_SMOKE_OUTPUT_DIR', '/content/drive/MyDrive/atom_runs/quick_test')
os.environ.setdefault('ATOM_FULL_OUTPUT_DIR', '/content/drive/MyDrive/atom_runs/run1')
os.environ.setdefault('ATOM_RESUME_CHECKPOINT_DIR', os.environ['ATOM_FULL_OUTPUT_DIR'])
os.environ.setdefault('ATOM_RESUME_START_GEN', '8')
os.environ.setdefault('ATOM_RESUME_TOTAL_GENS', '20')

# Auth defaults (public repo workflow)
os.environ.setdefault('ATOM_USE_GITHUB_TOKEN', '0')
if os.environ.get('ATOM_USE_GITHUB_TOKEN') != '1':
    os.environ.pop('GITHUB_TOKEN', None)

# Private repo option:
# import getpass
# os.environ['ATOM_USE_GITHUB_TOKEN'] = '1'
# os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub PAT: ').strip()
os.environ['ATOM_BRANCH'] = 'main'
for key in [
    'ATOM_REPO_URL',
    'ATOM_BRANCH',
    'ATOM_DRIVE_REPO',
    'ATOM_WORK_REPO',
    'ATOM_INSTALL_JAX_CUDA',
    'ATOM_JAX_VERSION',
    'ATOM_DRIVE_REPO_SYNC_MODE',
    'ATOM_USE_VMAP',
    'ATOM_SMOKE_OUTPUT_DIR',
    'ATOM_FULL_OUTPUT_DIR',
    'ATOM_RESUME_CHECKPOINT_DIR',
    'ATOM_RESUME_START_GEN',
    'ATOM_RESUME_TOTAL_GENS',
    'ATOM_USE_GITHUB_TOKEN',
]:
    print(f"{key}={os.environ[key]}")
print('GITHUB_TOKEN set =', 'GITHUB_TOKEN' in os.environ)

# set branch to main




## 3) Clone (if needed), preflight, and bootstrap

This handles stale partial clones, validates configuration, and then runs `colab_bootstrap.sh`.



In [ ]:
%%bash
set -euo pipefail

WORK_REPO="${ATOM_WORK_REPO:-/content/atom}"
REPO_URL="${ATOM_REPO_URL:-}"
BRANCH="${ATOM_BRANCH:-main}"
AUTH_REPO_URL="$REPO_URL"

echo "Using WORK_REPO=$WORK_REPO"
echo "Using BRANCH=$BRANCH"
echo "ATOM_DRIVE_REPO_SYNC_MODE = $ATOM_DRIVE_REPO_SYNC_MODE"

if [[ -z "$REPO_URL" ]]; then
  echo "ERROR: ATOM_REPO_URL must be set before first clone."
  exit 1
fi

if [[ "$REPO_URL" == *"<org>"* || "$REPO_URL" == *"<repo>"* ]]; then
  echo "ERROR: ATOM_REPO_URL still contains placeholders. Set a real repo URL first."
  exit 1
fi

if [[ "${ATOM_USE_GITHUB_TOKEN:-0}" == "1" && -n "${GITHUB_TOKEN:-}" && "$REPO_URL" == https://github.com/* ]]; then
  AUTH_REPO_URL="${REPO_URL/https:\/\//https:\/\/${GITHUB_TOKEN}@}"
fi

if [[ -d "$WORK_REPO" && ! -d "$WORK_REPO/.git" ]]; then
  echo "Found stale non-git directory at $WORK_REPO. Removing it..."
  rm -rf "$WORK_REPO"
fi

if [[ ! -d "$WORK_REPO/.git" ]]; then
  echo "Cloning repository..."
  if ! git clone --branch "$BRANCH" --single-branch "$AUTH_REPO_URL" "$WORK_REPO"; then
    echo "ERROR: git clone failed."
    echo "Common causes:"
    echo "  1) Wrong ATOM_REPO_URL"
    echo "  2) Branch does not exist remotely"
    echo "  3) Private repo without token"
    echo "  4) Temporary GitHub/network issue"
    exit 1
  fi
fi

cd "$WORK_REPO"
python -u -m src.training.utils.colab_preflight --stage bootstrap --strict

chmod +x colab_bootstrap.sh
bash colab_bootstrap.sh



## 4) Sanity checks


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python - <<'PY'
import torch
from src.training.utils.runtime_platform import detect_runtime_platform
print('torch.cuda.is_available =', torch.cuda.is_available())
print('detected runtime platform =', detect_runtime_platform())
PY

nvidia-smi || true


## 5) Quick infrastructure smoke test (streaming)

This validates wiring and streams logs live. Quick mode may fail graduation by design.



In [ ]:
import os
import shlex
import subprocess

work_repo = os.environ.get("ATOM_WORK_REPO", "/content/atom")
smoke_output_dir = os.environ.get("ATOM_SMOKE_OUTPUT_DIR", "/content/drive/MyDrive/atom_runs/quick_test")
use_vmap = os.environ.get("ATOM_USE_VMAP", "1") == "1"


def run_streaming(cmd, *, cwd, label, raise_on_error=True):
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"[{label}] $ {printable}")
    proc = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
    finally:
        rc = proc.wait()

    print(f"\n{label} exit code: {rc}")
    if raise_on_error and rc != 0:
        raise RuntimeError(f"{label} failed with exit code {rc}")
    return rc


preflight_cmd = [
    "python", "-u", "-m", "src.training.utils.colab_preflight",
    "--stage", "smoke",
    "--output-dir", smoke_output_dir,
    "--strict",
]
if use_vmap:
    preflight_cmd.append("--require-gpu")
run_streaming(preflight_cmd, cwd=work_repo, label="smoke preflight", raise_on_error=True)

train_cmd = [
    "python", "-u", "train_progressive.py",
    "--mode", "quick",
    "--device", "auto",
    "--output-dir", smoke_output_dir,
]
if use_vmap:
    train_cmd.append("--use-vmap")

run_streaming(train_cmd, cwd=work_repo, label="quick smoke", raise_on_error=False)



## 6) Full training run (real run, streaming)



In [ ]:
import os
import shlex
import subprocess

work_repo = os.environ.get("ATOM_WORK_REPO", "/content/atom")
full_output_dir = os.environ.get("ATOM_FULL_OUTPUT_DIR", "/content/drive/MyDrive/atom_runs/run1")
use_vmap = os.environ.get("ATOM_USE_VMAP", "1") == "1"


def run_streaming(cmd, *, cwd, label, raise_on_error=True):
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"[{label}] $ {printable}")
    proc = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
    finally:
        rc = proc.wait()

    print(f"\n{label} exit code: {rc}")
    if raise_on_error and rc != 0:
        raise RuntimeError(f"{label} failed with exit code {rc}")
    return rc


preflight_cmd = [
    "python", "-u", "-m", "src.training.utils.colab_preflight",
    "--stage", "full",
    "--output-dir", full_output_dir,
    "--strict",
]
if use_vmap:
    preflight_cmd.append("--require-gpu")
run_streaming(preflight_cmd, cwd=work_repo, label="full preflight", raise_on_error=True)

train_cmd = [
    "python", "-u", "train_progressive.py",
    "--mode", "complete",
    "--device", "auto",
    "--timesteps", "2000000",
    "--population", "8",
    "--generations", "10",
    "--output-dir", full_output_dir,
]
if use_vmap:
    train_cmd.append("--use-vmap")

run_streaming(train_cmd, cwd=work_repo, label="full training", raise_on_error=True)



## 7) Resume interrupted run (streaming)



In [ ]:
import os
import shlex
import subprocess

work_repo = os.environ.get("ATOM_WORK_REPO", "/content/atom")
checkpoint_dir = os.environ.get("ATOM_RESUME_CHECKPOINT_DIR", "/content/drive/MyDrive/atom_runs/run1")
start_gen = os.environ.get("ATOM_RESUME_START_GEN", "8")
total_gens = os.environ.get("ATOM_RESUME_TOTAL_GENS", "20")
use_vmap = os.environ.get("ATOM_USE_VMAP", "1") == "1"


def run_streaming(cmd, *, cwd, label, raise_on_error=True):
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"[{label}] $ {printable}")
    proc = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
    finally:
        rc = proc.wait()

    print(f"\n{label} exit code: {rc}")
    if raise_on_error and rc != 0:
        raise RuntimeError(f"{label} failed with exit code {rc}")
    return rc


preflight_cmd = [
    "python", "-u", "-m", "src.training.utils.colab_preflight",
    "--stage", "resume",
    "--checkpoint-dir", checkpoint_dir,
    "--strict",
]
if use_vmap:
    preflight_cmd.append("--require-gpu")
run_streaming(preflight_cmd, cwd=work_repo, label="resume preflight", raise_on_error=True)

resume_cmd = [
    "python", "-u", "scripts/training/resume_population_training.py",
    "--checkpoint-dir", checkpoint_dir,
    "--start-gen", str(start_gen),
    "--total-gens", str(total_gens),
]
if use_vmap:
    resume_cmd.append("--use-vmap")

run_streaming(resume_cmd, cwd=work_repo, label="resume run", raise_on_error=True)



In [ ]:
%%bash
set -euo pipefail
python -m pip install -U funkybob


## Troubleshooting

- Public repo: keep `ATOM_USE_GITHUB_TOKEN=0`.
- Private repo: set `ATOM_USE_GITHUB_TOKEN=1` and provide `GITHUB_TOKEN`.
- If Drive cache has local edits, bootstrap auto-stashes by default (`ATOM_DRIVE_REPO_SYNC_MODE=stash`).
- Use `ATOM_DRIVE_REPO_SYNC_MODE=reset` to discard cache edits, or `skip_pull` to avoid pulling when dirty.
- Use `python -m src.training.utils.colab_preflight --stage <bootstrap|smoke|full|resume> --strict` for fast diagnostics.
- Milestone gate checklist: `docs/COLAB_VALIDATION_CHECKLIST.md`.

